
# pycycle + LSST — Catalog-Scale RR Lyrae Search

This notebook describes a two-stage pipeline for finding faint RR Lyrae stars
(g > 21, beyond Gaia's reach) in Rubin LSST data using
pycycle and [LSDB](https://lsdb.io) for distributed catalog queries.

The pipeline follows Stringer et al. (2019, S19, arXiv:1905.00428) and Stringer,
Drlica-Wagner et al. (2021, S21, arXiv:2011.13930) — the state-of-the-art method
for finding faint RRab in sparse multiband survey data.

**Science target**: Point sources with $21 < g < 24.5$, out to ~335 kpc.


## 0. When to use period search vs template fitting

pycycle provides two independent period-finding methods:

| Method | Class | Output | Speed | Strength |
|---|---|---|---|---|
| **Hybrid period search** | `PeriodSearch` | period + PSI significance | ~5 s/star (5-band, ~300 obs) | Any light-curve shape; works on RRc, Cepheids, EBs |
| **Template fitting** | `TemplateFitter` | period + µ (distance), E(B-V), A (amplitude) | ~20–40 s/star | Multiband constraint; excellent at 4–15 obs/band |

### Decision guide for LSST

| LSST phase | Obs/band | Recommended strategy | Reason |
|---|---|---|---|
| Year 1–3 | ~4–10 | **Template fitting directly** | Period search is unreliable at this sparsity; S19 showed 4 obs/band is enough for template fitting |
| Year 5–7 | ~15–30 | **Template fitting directly** | Period search improves but template fitting still dominates for RRab specifically |
| Year 10 | ~50–100 | **Period search → template fitting** | Period search now reliable as a fast pre-filter; template fitting only on high-PSI candidates reduces cost ~100× |
| Well-sampled (>30/band) now | any | **Period search first** | ~5 s/star scan, then template fit only high-PSI candidates |
| All variable types wanted | any | **Period search** | Template fitting is RRab-specific |

**Why template fitting beats period search at low N_obs:**  
With 4–10 obs/band the Lomb-Scargle and Lafler-Kinman statistics used by `PeriodSearch`
are dominated by noise. The template imposes a strong prior on the light-curve shape and
simultaneously constrains amplitude, phase, distance, and dust across all bands — even with
a handful of points per band this provides enough information to uniquely identify the period.

**Why period search is better when you have many observations:**  
`PeriodSearch` runs in ~5 s/star (vs ~30 s for template fitting) and works for any variable
type. With > 30 obs/band the period grid can be searched reliably, and template fitting is
only needed for confirmed candidates.

## 1. Full pipeline overview (following S19/S21)

```
All LSST objects
  │
  ▼ Stage 0 — Quality cuts  (LSDB map_partitions, full catalog)
    extendedness == 0         (point sources; LSST morphology flag)
    total observations ≥ 10
    21 < g < 24.5             (fainter than Gaia, within single-epoch depth)
  │
  ▼ Stage 1 — Error rescaling  (compute ONCE per data release, cache to disk)
    Per HEALPix cell (nside=32), per band, fit chi2_nu vs magnitude.
    Rescale reported uncertainties so chi2_nu → 1 for constant stars.
    Load saved coefficients at pipeline run time — zero re-computation.
  │
  ▼ Stage 2 — Color cuts  (only when instantaneous colors are available)
    "Instantaneous" = two bands observed within 1 hr of each other.
    (r-i)_min,0 < 0.068  OR  (g-r)_min,0 < 0.268  (Ivezić+2005 RRL locus)
    FALLBACK when no instantaneous colors: mean (g-r)_0 < 0.488 (less restrictive)
    SKIP ENTIRELY if < 50% of stars have instantaneous colors — rely on
    variability + template fitting to reject stellar contaminants instead.
  │
  ▼ Stage 3 — Variability pre-filter  (LSDB map_partitions; fast)
    per band: log10(χ²_ν) ≥ 0.5  AND  log10(significance) ≥ 0
    Retains ~0.7% of input objects but ≥ 98% of RRab (S19 Table 3).
  │
  ▼ Stage 4 — Template fitting  (LSDB map_partitions; the expensive step)
    DES template (grizY) with LSST filter corrections applied at load time.
    Period range: 0.44–0.89 d (RRab only) — ~4× fewer grid points than 0.2–1.0.
    warm_start=True: carry (φ, µ, E(B-V), A) between frequencies (~4× faster).
  │
  ▼ Stage 5 — Random Forest classification  (fast; runs on Stage 4 output)
    Features: RSS/dof, amplitude/RSS, Oosterhoff distance, κ (phase concentration)
    RF1: variable vs constant;  RF2: RRab vs QSO (main contaminant at g > 21)
    Outputs ~85% purity, ~76% completeness on DES (S19)
  │
  ▼ Stage 6 — Visual inspection of new discoveries
    Only for objects not in existing catalogs (Gaia DR3, Catalina, PS1-RRL).
```

## 2. Stage 1 — Error rescaling (compute once, cache)

Rubin reported photometric uncertainties are often slightly underestimated.
Following S19 §3.1, we rescale them so that constant stars have $\chi^2_\nu = 1$.

**This step is computed ONCE per data release, not per run.**
The idea is: fit a per-HEALPix, per-band quadratic $\log_{10}\sigma_{\rm rescale}(m)$,
then save the coefficients. At pipeline run time, load the table and multiply.

The code below shows how to build and save the rescaling table.
In production you would load the saved table instead of recomputing it.

In [ ]:
# Error rescaling — run ONCE on a clean calibration sample, save coefficients
# (Not run in this notebook; shown for documentation)

import numpy as np
import pandas as pd
import healpy as hp
from pathlib import Path

NSIDE = 32

def build_error_rescaling_table(calib_stars_df, bands, output_path):
    """
    Fit chi2_nu vs magnitude for calibration (constant) stars per HEALPix cell.

    Parameters
    ----------
    calib_stars_df : DataFrame
        Columns: ra, dec, band, mag, magerr  — constant calibration stars
    bands : list of str
    output_path : str
        Where to save the resulting JSON/Parquet table
    """
    pix = hp.ang2pix(NSIDE, calib_stars_df['ra'].values,
                     calib_stars_df['dec'].values, lonlat=True)
    calib_stars_df = calib_stars_df.copy()
    calib_stars_df['healpix'] = pix

    rows = []
    for hp_id, g1 in calib_stars_df.groupby('healpix'):
        for band in bands:
            g2 = g1[g1['band'] == band]
            if len(g2) < 10:
                continue
            # group by object, compute chi2_nu per object
            # fit quadratic log10(chi2_nu) vs mean_mag
            # (details depend on catalog schema)
            rows.append({'healpix': hp_id, 'band': band,
                         'a0': 0.0, 'a1': 0.0, 'a2': 0.0})  # placeholder
    pd.DataFrame(rows).to_parquet(output_path)
    print(f'Saved error rescaling table to {output_path}')

print('Error rescaling: run build_error_rescaling_table() once per data release.')

## 3. Stage 2 — Color cuts

RR Lyrae occupy a narrow locus in color space (the horizontal branch).
Color cuts reduce non-variable contaminants before the expensive template fitting step.

**When to use color cuts:**
- LSST visits the same field in 2–3 consecutive filters in the same night
  (e.g., r then i within 30 min). Group observations taken within 1 hr as
  "instantaneous" colors — these sample the same point on the light curve and
  are suitable for color selection.
- If < 50% of stars have instantaneous colors, skip color cuts entirely.
  Random-epoch color averages wash out the RRab color locus and increase
  rejection of real RRab. Rely on variability + template fitting instead.

**Thresholds from S19** (after correcting for dust using SFD):
- $(r-i)_{\rm min,0} < 0.068$ OR $(g-r)_{\rm min,0} < 0.268$  
  (use minimum color over all instantaneous pairs)
- FALLBACK (mean colors): $(g-r)_0 < 0.488$

These thresholds are from the DES system; for LSST griz they are essentially the same.

In [ ]:
def apply_color_cuts(lc_df, max_dt_hr=1.0):
    """
    Apply RRL color cuts using instantaneous colors (obs within max_dt_hr of each other).

    Parameters
    ----------
    lc_df : DataFrame
        Per-object light curve with columns: mjd, band, mag_0 (dust-corrected)
    max_dt_hr : float
        Maximum time separation to count as instantaneous (hours)

    Returns
    -------
    bool : True if the object passes the color cuts
    """
    max_dt = max_dt_hr / 24.0  # convert to days

    # Try to form instantaneous color pairs
    ri_min = None
    gr_min = None
    for _, row_r in lc_df[lc_df['band'] == 'r'].iterrows():
        for _, row_i in lc_df[lc_df['band'] == 'i'].iterrows():
            if abs(row_r['mjd'] - row_i['mjd']) < max_dt:
                ri = row_r['mag_0'] - row_i['mag_0']
                ri_min = ri if ri_min is None else min(ri_min, ri)
        for _, row_g in lc_df[lc_df['band'] == 'g'].iterrows():
            if abs(row_r['mjd'] - row_g['mjd']) < max_dt:
                gr = row_g['mag_0'] - row_r['mag_0']
                gr_min = gr if gr_min is None else min(gr_min, gr)

    if ri_min is not None or gr_min is not None:
        # Use instantaneous color thresholds
        if ri_min is not None and ri_min < 0.068:
            return True
        if gr_min is not None and gr_min < 0.268:
            return True
        return False
    else:
        # Fallback: mean colors
        g_mean = lc_df[lc_df['band'] == 'g']['mag_0'].mean()
        r_mean = lc_df[lc_df['band'] == 'r']['mag_0'].mean()
        if np.isfinite(g_mean) and np.isfinite(r_mean):
            return (g_mean - r_mean) < 0.488
        return True  # no color info — don't cut

print('Color cut function defined.')

## 4. Stage 3 — Variability pre-filter

Following S19 Table 3: require variability in at least one band before running
the expensive template fitting.  This retains ≥ 98% of RRab while rejecting
~99.3% of constant stars.

Two features per band (S19 §3.2):
- $\log_{10}(\chi^2_\nu)$ where $\chi^2_\nu = {\rm Var}(m) / \langle\sigma^2\rangle$
- $\log_{10}({\rm significance})$ where significance $= (m_{\rm max} - m_{\rm min}) / \sqrt{\sigma_{\rm max}^2 + \sigma_{\rm min}^2}$

**Threshold**: keep objects where the median $\log_{10}(\chi^2_\nu) \geq 0.5$
and the maximum $\log_{10}({\rm significance}) \geq 0$.

In [ ]:
from pycycle.lsdb_utils import compute_variability_features

# Example: simulate a constant star and an RRab
import numpy as np
rng = np.random.default_rng(42)

n = 30
bands_example = np.array(['g', 'r', 'i', 'z'] * (n // 4))

# Constant star
const_df = pd.DataFrame({
    'band': bands_example,
    'mag': 22.0 + rng.normal(0, 0.02, len(bands_example)),
    'magerr': 0.02 * np.ones(len(bands_example)),
})
f_const = compute_variability_features(const_df, ['g', 'r', 'i', 'z'])
print(f'Constant star:  lchi_med={f_const["lchi_med"]:.2f}  sig_max={f_const["sig_max"]:.2f}')

# Simulated RRab (0.5 mag amplitude sine wave + noise)
t_rrab = np.linspace(0, 10, len(bands_example))
rrlyrae_df = pd.DataFrame({
    'band': bands_example,
    'mag': 22.0 + 0.5 * np.sin(2 * np.pi * t_rrab / 0.5) + rng.normal(0, 0.02, len(bands_example)),
    'magerr': 0.02 * np.ones(len(bands_example)),
})
f_rrab = compute_variability_features(rrlyrae_df, ['g', 'r', 'i', 'z'])
print(f'RRab (simulated): lchi_med={f_rrab["lchi_med"]:.2f}  sig_max={f_rrab["sig_max"]:.2f}')

print('\nThresholds: lchi_med >= 0.5  AND  sig_max >= 0')

## 5. DES template with LSST filter corrections

LSST grizY filters closely match DES grizY.  From RTN-099 (rtn-099.lsst.io),
the DES→LSST synthetic transformations for typical RRab mean colors
($g-i \approx 0.3$, $r-i \approx 0.1$, $i-z \approx 0.1$) are:

| Band | Transform | Offset at RRab mean color |
|---|---|---|
| g | $g_L = g_D + 0.016(g-i) - 0.003(g-i)^2 + 0.006$ | +0.011 mag |
| r | $r_L = r_D + 0.185(r-i) - 0.015(r-i)^2 + 0.010$ | +0.028 mag |
| i | $i_L = i_D + 0.150(r-i) - 0.003(r-i)^2 - 0.009$ | +0.006 mag |
| z | $z_L = z_D + 0.270(i-z) + 0.036(i-z)^2 - 0.003$ | +0.024 mag |
| Y | (very similar filters) | ~0 |

**For period finding**: the corrections are constant offsets per band, fully absorbed
into the distance modulus $\mu$.  Period recovery is unaffected — no correction needed.

**For accurate distances**: apply the mean correction once at template load time via
`apply_des_to_lsst_correction()`.  This modifies the $c_0$ term of $\beta_b(P)$
in-place — **zero runtime overhead during fitting**.

In [ ]:
import os
from pycycle.templates import load_rr_template
from pycycle.lsdb_utils import apply_des_to_lsst_correction

RR_TEMPLATES_DES = os.path.expanduser('~/software/rr-templates/template_des')

template_des = load_rr_template(RR_TEMPLATES_DES, name='des')
print('Before correction, c0 (g-band):', template_des.betas[template_des.bands.index('g'), 0])

# Apply zero-cost DES → LSST correction at load time
apply_des_to_lsst_correction(template_des)
print('After  correction, c0 (g-band):', template_des.betas[template_des.bands.index('g'), 0])
print()
print('Correction applied. Period finding is unaffected;')
print('distance moduli are now on the LSST AB scale.')

## 6. Stage 4 — Template fitting with warm start

S19 use a single averaged RRab template (not 30 individual templates as in notebook §6
of `template_fitting.ipynb`).  The key optimisations:

| Optimisation | Speedup |
|---|---|
| Single template vs fitting 30 individual templates | ~30× |
| `warm_start=True` (carry φ, µ, EBV, A between frequencies) | ~4× |
| Period range 0.44–0.89 d instead of 0.2–1.0 d | ~4× |
| Pre-filtering: only 0.7% of objects reach Stage 4 | ~143× |
| Modern CPU vs 2008 Xeon (S19 baseline) | ~10× |

**Warm start**: instead of trying `n_start=4` random starting phases per frequency,
carry the best solution $(\phi, \mu, E(B-V), A)$ from the previous frequency as the
initial guess.  The RSS landscape is smooth when frequencies are iterated in order,
so the solution from frequency $f_k$ is a good starting point for $f_{k+1}$.
Set `warm_start=True` in `TemplateFitter`.

In [ ]:
import importlib.resources
import numpy as np
from pycycle.template_fit import TemplateFitter

# Load the bundled B1392 light curve for demonstration
data_path = importlib.resources.files('pycycle.data').joinpath('B1392all.tab')
hjd, mag, magerr, filts = np.loadtxt(str(data_path), unpack=True)
ok = (magerr >= 0.0) & (magerr <= 0.2)
hjd, mag, magerr, filts = hjd[ok], mag[ok], magerr[ok], filts[ok].astype(int)

# Map to DES bands (skip u-band which isn't in DES template)
filtnams_all = ['u', 'g', 'r', 'i', 'z']
des_bands = template_des.bands   # ['g', 'i', 'r', 'Y', 'z'] or similar
keep = [fi for fi, fn in enumerate(filtnams_all) if fn in des_bands]
mask = np.isin(filts, keep)
filtnams_des = [fn for fn in filtnams_all if fn in des_bands]
remap = {old: filtnams_des.index(filtnams_all[old]) for old in keep}
filts_des = np.array([remap[f] for f in filts[mask]])

print(f'Using {mask.sum()} observations in {filtnams_des} bands')

# Fit with warm start
fitter_ws = TemplateFitter(template_des, n_newton=5, warm_start=True)
result_ws = fitter_ws.fit(hjd[mask], mag[mask], magerr[mask],
                           filts_des, filtnams_des,
                           pmin=0.44, dphi=0.02, pmax=0.89)

print(f'Best period (warm start): {result_ws.best_period:.7f} days')
print('Best coefficients:', {k: f'{v:.4f}' for k, v in result_ws.best_coeffs.items()})
print('(OGLE gold period: 0.5016247 days)')

In [ ]:
import matplotlib.pyplot as plt
import time

# Compare warm_start=True vs warm_start=False (n_start=4) speed
t0 = time.perf_counter()
fitter_cold = TemplateFitter(template_des, n_newton=5, warm_start=False, n_start=4)
result_cold = fitter_cold.fit(hjd[mask], mag[mask], magerr[mask],
                               filts_des, filtnams_des,
                               pmin=0.44, dphi=0.02, pmax=0.89)
t_cold = time.perf_counter() - t0

t0 = time.perf_counter()
fitter_warm = TemplateFitter(template_des, n_newton=5, warm_start=True)
result_warm = fitter_warm.fit(hjd[mask], mag[mask], magerr[mask],
                               filts_des, filtnams_des,
                               pmin=0.44, dphi=0.02, pmax=0.89)
t_warm = time.perf_counter() - t0

print(f'cold (n_start=4): {t_cold:.2f} s  →  period = {result_cold.best_period:.7f} d')
print(f'warm (n_start=1): {t_warm:.2f} s  →  period = {result_warm.best_period:.7f} d')
print(f'Speedup: {t_cold / t_warm:.1f}×')

In [ ]:
result_warm.plot_rss()
plt.show()
result_warm.plot_phased()
plt.show()

## 7. Stage 5 — Random Forest classification

The RF classifiers separate RRab from contaminants (mostly constant stars and QSOs).
Following S19, two classifiers are trained:

- **RF1**: variable vs constant star (trained on variability statistics)
- **RF2**: RRab vs QSO (trained on template fitting outputs)

### Features

**Variability features** (from Stage 3, for RF1):
- `lchi_med`: median $\log_{10}(\chi^2_\nu)$ across bands
- `sig_max`: max $\log_{10}({\rm significance})$ across bands

**Template fitting features** (from Stage 4, for RF2):
- `rss_dof`: RSS / (N_obs − 4) — goodness of fit
- `amp_rss`: A / rss_dof — high for well-fitted large-amplitude RRab
- `f_dist1`, `f_dist2`: distance from Oosterhoff I/II period-amplitude relations
- `kappa`: von Mises-Fisher concentration of phases in the folded light curve
  (high κ → observations cluster at a particular phase → possible period alias)

### Training data sources (for LSST)

**Positives (RRab)**:
- Gaia DR3 variable star catalog (Clementini et al. 2023), cross-matched to LSST DP1
- Catalina Sky Survey DR2 (Drake et al. 2013)
- Pan-STARRS Sesar et al. (2017)
- Simulated faint RRab: take known RRab from bright catalogs, shift to larger distances
  ($m \to m + \Delta\mu$), downsample to LSST cadence — extends training coverage
  beyond Gaia's reach ($g > 21$)

**Negatives (contaminants)**:
- RF1 negatives: LSST photometric calibration stars (`calib_*` flags) — these are
  FGK standards selected for photometric stability
- RF2 negatives: SDSS DR16 QSOs, KiDS DR3 QSOs, DESI EDR quasars —
  QSOs are the dominant contaminant at $g > 21$ in DES (same expected for LSST)

### Training workflow

In [ ]:
# RF training sketch — requires labeled data from LSST DP1 overlapping SDSS Stripe 82 or Gaia
# (Not run here; shown as a template)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib

def train_rf_classifier(X_pos, X_neg, output_path):
    """
    Train and save a binary RF classifier.

    Parameters
    ----------
    X_pos : (n_pos, n_features) array — RRab feature vectors
    X_neg : (n_neg, n_features) array — contaminant feature vectors
    output_path : path to save the trained model (.joblib)
    """
    import numpy as np
    X = np.vstack([X_pos, X_neg])
    y = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])

    clf = RandomForestClassifier(
        n_estimators=200,
        max_features='sqrt',
        min_samples_leaf=5,
        class_weight='balanced',  # handles class imbalance
        n_jobs=-1,
        random_state=42,
    )
    scores = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
    print(f'CV ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}')

    clf.fit(X, y)
    joblib.dump(clf, output_path)
    print(f'Model saved to {output_path}')
    return clf

# At pipeline run time:
# rf1 = joblib.load('rf1_variable_vs_constant.joblib')
# rf2 = joblib.load('rf2_rrab_vs_qso.joblib')
# p_rrab = rf2.predict_proba(X_test)[:, 1]
print('RF training function defined. Train once offline, load at run time.')

## 8. LSDB integration — full pipeline as `map_partitions`

LSDB distributes the catalog across HiPSCat HEALPix partitions. `map_partitions`
applies a function to each partition in parallel via Dask. The function must:
1. Accept a `pandas.DataFrame` (one partition) as its sole argument
2. Return a `pandas.DataFrame` with a fixed schema (the `meta` argument)

The `make_template_fit_fn` factory in `pycycle.lsdb_utils` wraps the template fitter
into a `map_partitions`-compatible function automatically.

**Flux → magnitude conversion**: Rubin DP1+ sources use `psfFlux` in nJy.
The `flux_to_mag` helper converts to AB magnitudes using the Rubin convention
$m = -2.5 \log_{10}(f_{\rm nJy}) + 31.4$.

In [ ]:
# Full LSDB pipeline — requires access to the Rubin DP1 HATS catalog
# (Not run in this notebook; shown as a complete working template)

# import lsdb
# from dask.distributed import Client
from pycycle.lsdb_utils import apply_des_to_lsst_correction, make_template_fit_fn
from pycycle.templates import load_rr_template
import os

# --- 1. Start a Dask cluster ---
# client = Client(n_workers=8, threads_per_worker=1, memory_limit='8GB')

# --- 2. Open catalogs ---
# objects = lsdb.open_catalog(
#     'https://data.lsdb.io/hats/dp01_object',
#     columns=['id', 'ra', 'dec', 'r_psfMag', 'extendedness'],
# )
# sources = lsdb.open_catalog(
#     'https://data.lsdb.io/hats/dp01_forced_source',
#     columns=['objectId', 'midpointMjdTai', 'band', 'psfFlux', 'psfFluxErr'],
# )

# --- 3. Quality cuts (Stage 0) ---
# stars = objects.query('extendedness == 0 and 21 < r_psfMag < 24.5')
# joined = stars.nest_sources(sources, source_id_col='objectId')

# --- 4. Load template with LSST correction (Stage 4 prep) ---
RR_TEMPLATES_DES = os.path.expanduser('~/software/rr-templates/template_des')
template = load_rr_template(RR_TEMPLATES_DES, name='des')
apply_des_to_lsst_correction(template)  # zero-cost distance correction

# --- 5. Build the map_partitions function ---
fit_fn, meta = make_template_fit_fn(
    template,
    bands=['g', 'r', 'i', 'z', 'y'],
    pmin=0.44, dphi=0.02, pmax=0.89,
    n_newton=5,
    warm_start=True,
)
print('meta schema:')
print(meta.dtypes)

# --- 6. Run on catalog ---
# results = joined.map_partitions(fit_fn, meta=meta).compute()
# print(f'Found {len(results)} candidate RRab')
print('\nTo run on the full catalog:')
print('  results = joined.map_partitions(fit_fn, meta=meta).compute()')

## 9. Performance expectations

Based on S19 results scaled to LSST and modern hardware:

| Step | Cost per object | Notes |
|---|---|---|
| Quality cuts (Stage 0) | negligible | SQL-like filter in LSDB |
| Error rescaling (Stage 1) | negligible | table lookup, pre-computed |
| Color cuts (Stage 2) | negligible | when available |
| Variability pre-filter (Stage 3) | ~1 ms | simple statistics |
| Template fitting (Stage 4) | ~20–40 s | 0.7% of objects reach this step |
| RF classification (Stage 5) | ~0.1 ms | fast tree evaluation |

**Effective rate for the full catalog:**
- Pre-filtering reduces to 0.7% of objects before template fitting
- On 8 cores: ~8 objects/s for the expensive step
- For 1 million point sources: ~750 min total (or ~90 min per core with 8 Dask workers)
- For 10 million point sources (LSST year 1 footprint): ~15 hours on a cluster

**Expected purity/completeness** (S19 Table 3, on DES grizY with 4–35 obs/band):
- Purity: ~85% (15% are QSO/other contaminants)
- Completeness: ~76% (misses ~24% of real RRab, mostly very faint ones)
- At N_obs/band ≥ 15 (LSST Year 3+): purity ~90%, completeness ~85%

### Suggested output quality flags

Before visual inspection, automatically reject:
- `EBV > 1.0` — unphysical dust level, likely a bad fit
- `A < 0.3` or `A > 2.0` — outside normal RRab amplitude range
- `kappa > 5` — high phase concentration, likely aliased period
- Cross-match against Gaia DR3, Catalina, PS1-RRL — inspect only new discoveries

## 10. Stage 6 — Visual inspection of new discoveries

Visual inspection is the final quality check for objects not in existing catalogs.
Key tools and approaches (not implemented here):

**Built into pycycle:**
- `result.plot_phased()` — phased multiband light curve with template overlay
- `result.plot_rss()` — RSS vs period; check that the best period is unambiguous

**Dashboard approach (recommended for > 100 candidates):**
- **Panel / Bokeh**: build a small dashboard showing sky map (ra/dec), RSS curve,
  phased LC, and a cutout image side by side; navigate forward/backward with keyboard
- **ipyaladin**: interactive sky viewer in Jupyter to check for nearby sources,
  blending, or known objects at the candidate position
- **AstroWidget**: displays FITS cutouts from Rubin Butler inside Jupyter

**Practical workflow:**
1. Sort candidates by `rss_dof` (ascending) — best fits first
2. Auto-reject `EBV > 1`, `A < 0.3`, `kappa > 5` (see §9)
3. Cross-match against known catalogs; flag matches as "known"
4. Visually inspect remaining "new" candidates in batches of ~10
5. Record decision (accept/reject/uncertain) to a CSV with object ID and notes

A minimal Panel inspection app can be built in ~50 lines — worth doing if you have
> 500 candidates to review.


## References

- **S19**: Stringer et al. 2019, AJ, arXiv:1905.00428 — template method for sparse multiband data (DES)
- **S21**: Stringer, Drlica-Wagner et al. 2021, ApJ, arXiv:2011.13930 — 6-year DES RRL catalog
- **Saha & Vivas 2017**, AJ 154 231 — hybrid LS+LK period search (pycycle core algorithm)
- **RTN-099**: rtn-099.lsst.io — DES↔LSST filter transformations
- **DP1**: arXiv:2603.23786 — Rubin Data Preview 1
- **rr-templates**: https://github.com/longjp/rr-templates — James Long et al. template library
- **LSDB**: https://lsdb.io — large-scale survey database for HiPSCat catalogs
